# Figure 5a - UMAP

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\StorywranglerData\RussiaUkraineJsons_19092023'
os.chdir(directory)

remove_UA_RU=False
smooth=True
Filter=False

In [ ]:
df = pd.read_csv(r'pivotUkraine_SUM_excelInput_weekly.csv') #, index_col=0, sep=';'
#Load frequency share pivoted df
df.set_index(df.columns[0], inplace=True)
df.head(3)

In [ ]:
# Filter out non-date columns: keep only columns that can be parsed as dates.
date_mask = pd.to_datetime(df.columns, errors='coerce').notna()
date_cols = df.columns[date_mask]
df_dates = df[date_cols].copy()

# Convert the remaining column names to datetime objects
df_dates.columns = pd.to_datetime(df_dates.columns)

# At this point, df_dates has languages as rows and dates as columns.
# To compute 6‑week sums, transpose so that dates become the index.
df_transposed = df_dates.T

# Group the rows (dates) by 6‑week periods and sum the values.
# Here, label='right' and closed='right' make the group label the last date of each 6‑week period.
df_6week_transposed = df_transposed.groupby(
    pd.Grouper(freq='6W', label='right', closed='right')
).sum()

# Transpose back so that rows are languages and columns represent 6‑week aggregated sums.
df = df_6week_transposed.T


In [ ]:
import umap
import pandas as pd
import plotly.graph_objects as go
import matplotlib.dates as mdates
from sklearn.manifold import TSNE

# Assume 'df' is your DataFrame with languages as rows and dates as columns,
# with one extra column (e.g., "language") that should be excluded.
# Filter out non-date columns:
date_mask = pd.to_datetime(df.columns, errors='coerce').notna()
date_cols = df.columns[date_mask]
df_dates = df[date_cols].copy()

# Convert the remaining column names to datetime objects
df_dates.columns = pd.to_datetime(df_dates.columns)

# Transpose the DataFrame so that each date becomes an observation (vector)
# Now: index = dates, columns = languages
df_transposed = df_dates.T

# Apply UMAP reduction with modified parameters:
# n_neighbors controls the local neighborhood size,
# min_dist controls how tightly points are packed,
# metric defines the distance metric used.


metric='euclidean'
graphtype='tsne'
variable=15

if graphtype=='umap':
    reducer = umap.UMAP(
        random_state=42,
        n_neighbors=variable,  # Adjust the number of neighbors (default is 15)
        min_dist=0.1,    # Adjust the minimum distance between points in the embedding
        metric=metric # Change the distance metric (default is 'euclidean')
    )
    embedding = reducer.fit_transform(df_transposed.values)  # shape: (n_dates, 2)
elif graphtype=='tsne':
    # You can adjust perplexity and n_iter based on your data and desired resolution
    tsne = TSNE(random_state=42, metric=metric, perplexity=variable, n_iter=1000)
    embedding = tsne.fit_transform(df_transposed.values)  # shape: (n_dates, 2)

# Create a DataFrame for the UMAP embedding; the index holds the dates.
umap_df = pd.DataFrame(embedding, columns=['UMAP1', 'UMAP2'], index=df_transposed.index)
umap_df.reset_index(inplace=True)
umap_df.rename(columns={'index': 'date'}, inplace=True)


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.dates as mdates
import pandas as pd

# Convert the dates to numeric values for the color mapping.
umap_df['date_numeric'] = mdates.date2num(umap_df['date'])

# Create custom tick values for the colorbar:
min_year = umap_df['date'].min().year
max_year = umap_df['date'].max().year
tick_dates = [pd.Timestamp(year=year, month=1, day=1) for year in range(min_year, max_year + 1)]
tickvals = [mdates.date2num(tick) for tick in tick_dates]
ticktext = [tick.strftime("%Y") for tick in tick_dates]

# Sort the DataFrame by date for the line traces.
umap_df_sorted = umap_df.sort_values(by='date')

# Define the colorscale and calculate normalization bounds.
colorscale = px.colors.sequential.Viridis
n_colors = len(colorscale)
date_min = umap_df_sorted['date_numeric'].min()
date_max = umap_df_sorted['date_numeric'].max()

# Create an interactive Plotly scatter plot.
fig = go.Figure()

# Add the markers with color mapping.
fig.add_trace(go.Scatter(
    y=umap_df['UMAP1'],
    x=umap_df['UMAP2'],
    mode='markers',
    showlegend=False,  # disable legend entry for the main scatter
    marker=dict(
        size=7,
        color=umap_df['date_numeric'],  # color mapped by numeric date
        colorscale='Viridis',
        colorbar=dict(
            title='Date',
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext
        ),
        showscale=True
    ),
    text=umap_df['date'].astype(str),
    hovertemplate='Date: %{text}<br>UMAP1: %{x:.2f}<br>UMAP2: %{y:.2f}<extra></extra>'
))

# Add connecting lines between consecutive points.
for i in range(1, len(umap_df_sorted)):
    date_val = umap_df_sorted.iloc[i]['date_numeric']
    norm = (date_val - date_min) / (date_max - date_min)
    color_index = int(norm * (n_colors - 1))
    color = colorscale[color_index]
    
    fig.add_trace(go.Scatter(
        y=[umap_df_sorted.iloc[i-1]['UMAP1'], umap_df_sorted.iloc[i]['UMAP1']],
        x=[umap_df_sorted.iloc[i-1]['UMAP2'], umap_df_sorted.iloc[i]['UMAP2']],
        mode='lines',
        line=dict(color=color, width=2),
        opacity=0.5,
        hoverinfo='skip',
        showlegend=False  # disable legend entry for the line traces
    ))

fig.update_layout(
    xaxis_title="Dimension 2",
    yaxis_title="Dimension 1",
    template="plotly_white",
    width=800,
    height=630
)

fig.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# Assume umap_df is your DataFrame and contains a 'date' column.
# Convert dates to numeric values for color mapping.
umap_df['date_numeric'] = mdates.date2num(umap_df['date'])

# Create custom tick values for the colorbar:
min_year = umap_df['date'].min().year
max_year = umap_df['date'].max().year

# Create tick dates as January 1st of each year in the range, excluding 2008.
tick_dates = [pd.Timestamp(year=year, month=1, day=1) 
              for year in range(min_year, max_year + 1) if year != 2008]
tickvals = [mdates.date2num(tick) for tick in tick_dates]
ticklabels = [tick.strftime("%Y") for tick in tick_dates]

# Sort the DataFrame by date so the connecting lines follow the correct order.
umap_df_sorted = umap_df.sort_values(by='date_numeric').reset_index(drop=True)

# Set normalization bounds based on date_numeric values.
date_min = umap_df_sorted['date_numeric'].min()
date_max = umap_df_sorted['date_numeric'].max()
norm = mcolors.Normalize(vmin=date_min, vmax=date_max)
cmap = plt.get_cmap('viridis')

# Create the scatter plot.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(umap_df['UMAP2'], umap_df['UMAP1'],
                     c=umap_df['date_numeric'],
                     cmap='viridis',
                     s=18)  # marker size

# Add connecting lines between consecutive points.
for i in range(1, len(umap_df_sorted)):
    # Use the date of the second point to choose the line color.
    date_val = umap_df_sorted.loc[i, 'date_numeric']
    color = cmap(norm(date_val))
    ax.plot([umap_df_sorted.loc[i-1, 'UMAP2'], umap_df_sorted.loc[i, 'UMAP2']],
            [umap_df_sorted.loc[i-1, 'UMAP1'], umap_df_sorted.loc[i, 'UMAP1']],
            color=color, linewidth=2, alpha=0.5)

# Increase axis label sizes.
ax.set_xlabel("Dimension 2", fontsize=11)
ax.xaxis.set_label_position("top")
ax.set_ylabel("Dimension 1", fontsize=11)

# Increase tick label size for both axes.
ax.tick_params(axis='both', labelsize=9)
ax.invert_yaxis()

# Add a colorbar with custom ticks and labels.
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_ticks(tickvals)
cbar.set_ticklabels(ticklabels)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# Define the output directory (modify this path as needed)
output_dir = r'D:\OneDrive\TLU\PhD RESEARCH\Artiklid\Ukraine-Twitter\Visualizations\Fig.5_DateVectors\UMAP\\'

# Create subdirectories for saving the files
os.makedirs(f"{output_dir}pdf", exist_ok=True)
os.makedirs(f"{output_dir}jpeg", exist_ok=True)
os.makedirs(f"{output_dir}svg", exist_ok=True)

# Get current date formatted as YearMonthDay (e.g., 20250314)
today_str = datetime.now().strftime("%Y%m%d")

# Use today_str in your filename; ensure 'metric' is defined or replace it with a string
fig.savefig(f"{output_dir}pdf/totalfreq_{graphtype}_{metric}_{variable}_{today_str}.pdf", format='pdf')
fig.savefig(f"{output_dir}jpeg/totalfreq_{graphtype}_{metric}_{variable}_{today_str}.jpg", format='jpg', dpi=500)
fig.savefig(f"{output_dir}svg/totalfreq_{graphtype}_{metric}_{variable}_{today_str}.svg", format='svg')
